In [1]:
!pip install langchain
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.4 MB/s eta 0:00:00


## Branching and Merging Chains with LCEL

The idea here is to have multiple branching LLM Chains which work independently in parallel and then we merge their outputs finally using a merge LLM chain at the end to get a consolidated output

In [2]:
from getpass import getpass
import os

OPENAI_KEY = getpass('Enter Open AI API Key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_KEY

Enter Open AI API Key: ··········


In [4]:
#generate LLM Instance
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model= 'gpt-4o-mini')

In [5]:
# generate some descripts about a topic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

description_prompt =  ChatPromptTemplate.from_template(
    """Generate a two line description for the given topic:
      {topic}
""")

description_chain = description_prompt | llm |StrOutputParser()

#generate pros about the topic

pro_prompt = ChatPromptTemplate.from_template(
    """Generate three bullet points talking about the pros for the given topic:
      {topic}
""")

pro_chain = pro_prompt | llm | StrOutputParser()


#generate cons prompt
con_prompt = ChatPromptTemplate.from_template(
    """Generate three bullet points talking about the cons for the given topic:
      {topic}
""")


con_chain = con_prompt | llm | StrOutputParser()



In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from operator import itemgetter

branch_chain = RunnableParallel(
    topic = itemgetter('topic'),  #itemgetter i used here so that we can store the topic given by user in the topic variable
    description = description_chain,
    pros = pro_chain,
    cons = con_chain
)

In [19]:
#this is another way of doing it if we do not want to use Itemgetter here
# branch_chain = RunnableParallel(
#     topic = RunnablePassthrough(),
#     description = description_chain,
#     pros = pro_chain,
#     cons = con_chain
# )

In [22]:
branch_chain.invoke("Artificial Intelligence")

{'topic': 'Artificial Intelligence',
 'description': 'Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems, enabling them to perform tasks such as learning, reasoning, and decision-making. It encompasses a wide range of technologies, including machine learning, natural language processing, and robotics, transforming industries and enhancing everyday life.',
 'pros': '- **Enhanced Efficiency and Productivity**: Artificial Intelligence can automate repetitive tasks and analyze large datasets at unprecedented speeds, allowing organizations to operate more efficiently and freeing up human resources for higher-value work.\n\n- **Improved Decision-Making**: AI systems can process vast amounts of information and identify patterns that may be missed by humans, leading to more informed and data-driven decision-making across various domains, from business strategy to healthcare.\n\n- **Personalization and User Experience

In [23]:
merge_prompt = ChatPromptTemplate.from_template(
    """Create a report about {topic} with the following information:
      Description:
      {description}
      Pros:
      {pros}
      Cons:
      {cons}

      Report should be in the following format:

      Topic: <name of the topic>

      Description: <description of the topic>

      Pros and Cons:

      <table with two columns showing the 3 pros and cons of the topic>
""")

merge_chain = merge_prompt | llm

In [24]:
final_chain = (branch_chain | merge_chain)

In [25]:
response = final_chain.invoke ({'topic':"Artificial Intelligence"})

In [26]:
from IPython.display import Markdown, display

display(Markdown(response.content))

# Report on Artificial Intelligence

---

**Topic:** Artificial Intelligence

**Description:**  
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines, enabling them to learn, reason, and solve problems autonomously. It encompasses various technologies, including machine learning, natural language processing, and robotics, transforming industries and enhancing everyday experiences.

---

**Pros and Cons:**

| **Pros**                                       | **Cons**                                         |
|------------------------------------------------|-------------------------------------------------|
| **Enhanced Efficiency and Productivity**       | **Job Displacement**                            |
| Artificial Intelligence can automate repetitive tasks, allowing businesses and individuals to focus on more complex and creative endeavors. This leads to increased productivity, reduced human error, and significant time savings across various industries. | The increasing adoption of AI technologies can lead to significant job losses across various sectors, as machines and algorithms become capable of performing tasks traditionally executed by humans, potentially widening economic inequalities. |
| **Data-Driven Insights and Decision Making**   | **Ethical Concerns**                            |
| AI can analyze vast amounts of data quickly and accurately, uncovering patterns and trends that humans might miss. This ability enables organizations to make informed, data-driven decisions, improving strategic planning and operational effectiveness. | The use of AI raises significant ethical issues, including biases in algorithmic decision-making, lack of accountability, and privacy concerns, especially when it comes to data handling and surveillance, potentially leading to discrimination and misuse of power. |
| **Personalization and Improved User Experience** | **Dependence on Technology**                    |
| AI technologies can offer personalized experiences in sectors like e-commerce, healthcare, and entertainment by analyzing user behavior and preferences. This customization enhances customer satisfaction, engagement, and loyalty, driving better business outcomes. | Over-reliance on AI systems may erode essential human skills and decision-making abilities, creating vulnerabilities in critical areas such as healthcare, transportation, and security if these systems fail or are compromised. |

--- 

This report highlights the transformative impact of Artificial Intelligence, presenting both its advantages in enhancing productivity and personalization, as well as concerns about job displacement, ethical implications, and dependency on technology.